In [65]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

### Reading and compacting dfs

In [66]:
compact = False

In [67]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir('dfs'):
    if file.endswith('.parquet'):
        read.append(file)
        df = pd.read_parquet(f"dfs/{file}")
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

if compact:
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = 'dfs/trash'
    for f in read:
        os.rename(f'dfs/{f}', f'{trash}/{f}')

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    df.to_parquet(f'dfs/compacted_{timestamp}.parquet')

df

,prob_P_has_hand_0,prob_P_has_hand_1,prob_P_has_hand_2,prob_P_has_hand_3,prob_P_has_hand_4,prob_P_has_hand_5,prob_P_has_hand_6,prob_P_has_hand_7,prob_P_has_hand_8,prob_P_has_hand_9,...,player_bet,player_bet_in_round,opponent_bet,opponent_bet_in_round,player_turn,player_has_bet,opponent_has_bet,pot,game_size,stage
0,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,...,0,41,0,41,True,True,True,82,200,terminal
1,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,...,41,82,41,82,False,True,True,164,200,terminal
2,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,...,0,41,41,82,True,True,True,123,200,river
3,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,...,3,44,3,44,False,True,True,88,200,terminal
4,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,0.000925,...,0,41,3,44,True,True,True,85,200,river
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6209,0.000000,0.000000,0.000000,0.008697,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0,89,3,92,True,True,True,181,200,river
6210,0.000000,0.000000,0.000000,0.006040,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,3,92,3,92,False,True,True,184,200,terminal
6211,0.000000,0.000000,0.000000,0.008697,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0,89,3,92,True,True,True,181,200,river
6212,0.000000,0.000000,0.000000,0.006056,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,3,92,3,92,False,True,True,184,200,terminal


### Preprocessing

In [63]:
# Divide all bets by game_size
bet_columns = [
    "player_bet",
    "player_bet_in_round",
    "opponent_bet",
    "opponent_bet_in_round",
    "pot",
]
df[bet_columns] = df[bet_columns].div(df["game_size"], axis=0)
df[bet_columns]

,player_bet,player_bet_in_round,opponent_bet,opponent_bet_in_round,pot
0,0.000,0.205,0.000,0.205,0.410
1,0.205,0.410,0.205,0.410,0.820
2,0.000,0.205,0.205,0.410,0.615
3,0.015,0.220,0.015,0.220,0.440
4,0.000,0.205,0.015,0.220,0.425
...,...,...,...,...,...
408,0.025,0.230,0.000,0.205,0.435
409,0.100,0.305,0.100,0.305,0.610
410,0.100,0.305,0.000,0.205,0.510
411,0.025,0.230,0.025,0.230,0.460


In [71]:
def to_X_and_Y(df, stage):
    df = df[df['stage'] == stage]
    Y_columns = [col for col in df.columns if col.startswith('value_of_hand_')]
    X_columns = [col for col in df.columns if col not in Y_columns]
    X = df[X_columns].values
    Y = df[Y_columns].values
    return X, Y

X_term, Y_term = to_X_and_Y(df, 'terminal')
print(X_term.shape, Y_term.shape)

(3006, 2715) (3006, 1326)


In [72]:
X_river, Y_river = to_X_and_Y(df, 'river')
print(X_river.shape, Y_river.shape)

(3208, 2715) (3208, 1326)
